# Dataset 1 (human-side) assembly

Merges all Core **human** rows from `v2/data/interim/gathered/*_prepared.jsonl` into a single JSONL for downstream Core assembly.

Spec: `v2/docs/dataset_design_final.md` §4.1 (required fields), §5 (human-side / Dataset 1).

**Out of scope here:** train/val/test splits and manifest (`split` stays as in gathered, typically `tbd`).

**HC3:** `financial_qa_prepared.jsonl` contains human + ChatGPT; this notebook keeps **human only** (`label == 0`).

In [10]:
from __future__ import annotations

import json
import re
import sys
import unicodedata
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

REPO = Path("/Users/askar/projects/antifraud-deepfake-detection")
BASE = REPO / "v2"
sys.path.insert(0, str(BASE))
from config import length_bin as compute_length_bin

GATHERED = BASE / "data" / "interim" / "gathered"
ASSEMBLED_DIR = BASE / "data" / "interim" / "assembled"
OUT_JSONL = ASSEMBLED_DIR / "dataset1_human.jsonl"
TABLES_DIR = BASE / "outputs" / "tables"

# Fixed merge order: first occurrence wins global dedup (§ assembly checklist).
INPUT_SPECS: list[tuple[str, str]] = [
    ("nazario_prepared.jsonl", "phishing_email"),
    ("nigerian_fraud_prepared.jsonl", "advance_fee_scam_email"),
    ("smishtank_prepared.jsonl", "fraud_sms_deceptive"),
    ("mendeley_smishing_prepared.jsonl", "fraud_sms_deceptive"),
    ("enron_ham_prepared.jsonl", "legitimate_email"),
    ("spamassassin_ham_prepared.jsonl", "legitimate_email"),
    ("sms_ham_prepared.jsonl", "legitimate_sms"),
    ("financial_qa_prepared.jsonl", "financial_qa"),
]

REQUIRED_KEYS = frozenset(
    {
        "text",
        "label",
        "label_str",
        "fraudness",
        "channel",
        "scenario_family",
        "source_family",
        "dataset_source",
        "time_band",
        "length_bin",
        "origin_model",
        "split",
    }
)

SCENARIO_WHITELIST = frozenset(
    {
        "phishing_email",
        "advance_fee_scam_email",
        "fraud_sms_deceptive",
        "legitimate_email",
        "legitimate_sms",
        "financial_qa",
    }
)

VALID_FRAUDNESS = frozenset({"fraud", "legitimate"})
VALID_CHANNEL = frozenset({"email", "sms", "qa"})

missing = [name for name, _ in INPUT_SPECS if not (GATHERED / name).is_file()]
if missing:
    raise FileNotFoundError(
        "Missing gathered files (run preparation notebooks first):\n  "
        + "\n  ".join(str(GATHERED / m) for m in missing)
    )

ASSEMBLED_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("GATHERED:", GATHERED)
print("OUT_JSONL:", OUT_JSONL)
print("TABLES_DIR:", TABLES_DIR)

GATHERED: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/gathered
OUT_JSONL: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/assembled/dataset1_human.jsonl
TABLES_DIR: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables


In [11]:
def normalize_text_for_dedup(text: str) -> str:
    """Same normalization idea as smishtank_prepare / enron_prepare (NFKC + whitespace)."""
    text = "" if text is None else str(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\uFFFD", " ")
    text = re.sub(r"[\x00-\x1F\x7F]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_jsonl(path: Path) -> list[dict]:
    rows: list[dict] = []
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
            rows[-1]["provenance_source_file"] = path.name
            rows[-1]["provenance_line_no"] = line_no
    return rows


per_file_counts: dict[str, int] = {}
all_rows: list[dict] = []
for fname, expected_sf in INPUT_SPECS:
    p = GATHERED / fname
    chunk = load_jsonl(p)
    per_file_counts[fname] = len(chunk)
    all_rows.extend(chunk)

total_loaded = len(all_rows)
print("Lines loaded per file:")
for k, v in per_file_counts.items():
    print(f"  {k:40s} {v:6d}")
print("TOTAL loaded:", total_loaded)

Lines loaded per file:
  nazario_prepared.jsonl                     5201
  nigerian_fraud_prepared.jsonl              3234
  smishtank_prepared.jsonl                    483
  mendeley_smishing_prepared.jsonl            369
  enron_ham_prepared.jsonl                  15088
  spamassassin_ham_prepared.jsonl            3753
  sms_ham_prepared.jsonl                     4061
  financial_qa_prepared.jsonl                7842
TOTAL loaded: 40031


In [12]:
# Human-only: drop LLM rows (critical for financial_qa_prepared.jsonl).
non_human_by_file: Counter[str] = Counter()
human_rows: list[dict] = []
for row in all_rows:
    src = row.get("provenance_source_file", "?")
    lbl = row.get("label")
    ls = row.get("label_str")
    if lbl != 0 or ls != "human":
        non_human_by_file[src] += 1
        continue
    human_rows.append(row)

print("Dropped non-human rows:", sum(non_human_by_file.values()))
if non_human_by_file:
    for k, v in sorted(non_human_by_file.items()):
        print(f"  {k}: {v}")
print("After human filter:", len(human_rows))

Dropped non-human rows: 3910
  financial_qa_prepared.jsonl: 3910
After human filter: 36121


In [13]:
# Validate §4.1 keys and value constraints.
issues: list[str] = []
expected_sf_by_file = {name: exp for name, exp in INPUT_SPECS}

for i, row in enumerate(human_rows):
    src = row.get("provenance_source_file", "?")
    missing_keys = sorted(REQUIRED_KEYS - row.keys())
    if missing_keys:
        issues.append(f"row_index={i} file={src} missing_keys={missing_keys}")
        continue

    text = row.get("text")
    if text is None or not str(text).strip():
        issues.append(f"row_index={i} file={src} empty_text")

    if row.get("fraudness") not in VALID_FRAUDNESS:
        issues.append(
            f"row_index={i} file={src} bad fraudness={row.get('fraudness')!r}"
        )

    if row.get("channel") not in VALID_CHANNEL:
        issues.append(f"row_index={i} file={src} bad channel={row.get('channel')!r}")

    sf = row.get("scenario_family")
    if sf not in SCENARIO_WHITELIST:
        issues.append(f"row_index={i} file={src} bad scenario_family={sf!r}")

    exp_sf = expected_sf_by_file.get(src)
    if exp_sf is not None and sf != exp_sf:
        issues.append(
            f"row_index={i} file={src} scenario_family={sf!r} expected={exp_sf!r}"
        )

if issues:
    print("VALIDATION FAILED:", len(issues), "issues (showing up to 25)")
    for msg in issues[:25]:
        print(" ", msg)
    raise ValueError("Fix gathered outputs or assembly rules before continuing.")

print("Schema validation: OK (", len(human_rows), "rows)")

Schema validation: OK ( 36121 rows)


In [14]:
# Optional diagnostic: length_bin vs config.py(token_length, channel)
mismatches: list[tuple[str, int, str, str, int]] = []
for row in human_rows:
    tl = row.get("token_length")
    if tl is None:
        continue
    ch = row["channel"]
    try:
        exp = compute_length_bin(int(tl), ch)
    except Exception:
        continue
    if row.get("length_bin") != exp:
        mismatches.append(
            (
                row.get("provenance_source_file", "?"),
                int(row.get("provenance_line_no", -1)),
                str(row.get("length_bin")),
                exp,
                int(tl),
            )
        )

if mismatches:
    print("WARNING: length_bin mismatches vs config.py:", len(mismatches))
    for t in mismatches[:10]:
        print(" ", t)
else:
    print("length_bin diagnostic: all checked rows match config.py")

length_bin diagnostic: all checked rows match config.py


In [15]:
# Global dedup on normalized `text` (order = INPUT_SPECS; first wins).
seen: dict[str, dict] = {}
dedup_dropped: list[dict] = []

for row in human_rows:
    key = normalize_text_for_dedup(row["text"])
    if not key:
        dedup_dropped.append(
            {
                "reason": "empty_norm_text",
                "dropped_file": row.get("provenance_source_file"),
                "dropped_line": row.get("provenance_line_no"),
            }
        )
        continue
    if key in seen:
        keeper = seen[key]
        dedup_dropped.append(
            {
                "reason": "duplicate_text",
                "kept_file": keeper.get("provenance_source_file"),
                "kept_line": keeper.get("provenance_line_no"),
                "dropped_file": row.get("provenance_source_file"),
                "dropped_line": row.get("provenance_line_no"),
            }
        )
        continue
    seen[key] = row

deduped_rows = list(seen.values())
# Preserve deterministic order: first-seen in human_rows iteration
order_index = {id(r): i for i, r in enumerate(human_rows)}
deduped_rows.sort(key=lambda r: order_index[id(r)])

print("After global dedup:", len(deduped_rows))
print("Dropped in dedup step:", len(dedup_dropped))

collisions = [d for d in dedup_dropped if d.get("reason") == "duplicate_text"]
TOP_N = 50
coll_sample_path = TABLES_DIR / "dataset1_human_dedup_collisions_sample.csv"
pd.DataFrame(collisions[:TOP_N]).to_csv(coll_sample_path, index=False)
print("Wrote collision sample:", coll_sample_path)

After global dedup: 36120
Dropped in dedup step: 1
Wrote collision sample: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset1_human_dedup_collisions_sample.csv


In [16]:
# Summary tables → v2/outputs/tables/
df = pd.DataFrame(deduped_rows)

def save_counts(series: pd.Series, name: str) -> Path:
    out = TABLES_DIR / name
    s = series.value_counts(dropna=False).rename("count").reset_index()
    s.columns = [series.name or "key", "count"]
    s.to_csv(out, index=False)
    return out


p1 = save_counts(df["scenario_family"], "dataset1_human_counts_by_scenario_family.csv")
p2 = df.groupby(["channel", "fraudness"], dropna=False).size().reset_index(name="count")
p2_path = TABLES_DIR / "dataset1_human_counts_by_channel_fraudness.csv"
p2.to_csv(p2_path, index=False)
p3 = save_counts(df["time_band"], "dataset1_human_counts_by_time_band.csv")
p3b = save_counts(df["length_bin"], "dataset1_human_counts_by_length_bin.csv")
p4 = df.groupby(["channel", "length_bin"], dropna=False).size().reset_index(name="count")
p4_path = TABLES_DIR / "dataset1_human_counts_by_channel_length_bin.csv"
p4.to_csv(p4_path, index=False)

run_ts = datetime.now(timezone.utc).isoformat()
summary = pd.DataFrame(
    [
        {
            "run_utc_iso": run_ts,
            "total_loaded_lines": total_loaded,
            "after_human_filter": len(human_rows),
            "after_global_dedup": len(deduped_rows),
            "dedup_dropped_total": len(dedup_dropped),
            "dedup_duplicate_text": len(collisions),
            "out_jsonl": str(OUT_JSONL),
        }
    ]
)
summary_path = TABLES_DIR / "dataset1_human_assembly_summary.csv"
summary.to_csv(summary_path, index=False)

print("Wrote:", p1)
print("Wrote:", p2_path)
print("Wrote:", p3)
print("Wrote:", p3b)
print("Wrote:", p4_path)
print("Wrote:", summary_path)

Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset1_human_counts_by_scenario_family.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset1_human_counts_by_channel_fraudness.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset1_human_counts_by_time_band.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset1_human_counts_by_length_bin.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset1_human_counts_by_channel_length_bin.csv
Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/outputs/tables/dataset1_human_assembly_summary.csv


In [17]:
# Write assembled JSONL (provenance keys kept for traceability).
with OUT_JSONL.open("w", encoding="utf-8") as out:
    for row in deduped_rows:
        out.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Wrote:", OUT_JSONL)
print("Final row count:", len(deduped_rows))

Wrote: /Users/askar/projects/antifraud-deepfake-detection/v2/data/interim/assembled/dataset1_human.jsonl
Final row count: 36120


In [18]:
# Printed summary (all numbers from this run)
print("=== Dataset 1 human assembly (this run) ===")
print("total_loaded_lines:", total_loaded)
print("after_human_filter:", len(human_rows))
print("after_global_dedup:", len(deduped_rows))
print("non_human_dropped:", sum(non_human_by_file.values()))
print("dedup_dropped_total:", len(dedup_dropped))
print()
print("By scenario_family:")
print(df["scenario_family"].value_counts().to_string())
print()
print("By channel × fraudness:")
print(df.groupby(["channel", "fraudness"]).size().to_string())

=== Dataset 1 human assembly (this run) ===
total_loaded_lines: 40031
after_human_filter: 36121
after_global_dedup: 36120
non_human_dropped: 3910
dedup_dropped_total: 1

By scenario_family:
scenario_family
legitimate_email          18841
phishing_email             5201
legitimate_sms             4061
financial_qa               3932
advance_fee_scam_email     3233
fraud_sms_deceptive         852

By channel × fraudness:
channel  fraudness 
email    fraud          8434
         legitimate    18841
qa       legitimate     3932
sms      fraud           852
         legitimate     4061
